# AST AudioSet — DIMER audio event classification tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/ast-audio-classification-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/ast-audio-classification-pipeline/blob/main/tutorials/ast_audio_classification_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-MIT%2Fast--finetuned--audioset-ffcc4d?style=flat)](https://huggingface.co/MIT/ast-finetuned-audioset-10-10-0.4593) [![Upstream](https://img.shields.io/badge/Upstream-YuanGongND%2Fast-181717?style=flat&logo=github&logoColor=white)](https://github.com/YuanGongND/ast) [![arXiv](https://img.shields.io/badge/arXiv-2104.01778-b31b1b.svg)](https://arxiv.org/abs/2104.01778)

**Profile:** `TASK-INFERENCE`  
**Notebook specification:** DIMER Notebook Specification 1.1 — **standalone** (§3.6)  
**Capability:** multi-label audio event classification over the 527 AudioSet labels using the pinned `MIT/ast-finetuned-audioset-10-10-0.4593` weights

**This notebook is standalone.** It carries the repository's pipeline module (`src/ast_audio_classification_pipeline/pipeline.py` at revision `98e26f9d5e48`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `f826b80d28226b62986cc218e5cec390b1096902` (~346 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/1); edit the repository and regenerate rather than editing cells.

At inference the clip is resampled to 16 kHz, turned into a 128-bin Kaldi filterbank with a 10 ms hop, padded or cropped to the model's 1024-frame (10.24 s) window, and passed through the spectrogram transformer; the pipeline applies an independent sigmoid per label and returns the top-k labels with their scores. **No adaptation occurs:** no training, fine-tuning, in-context conditioning, or preprocessing fitting happens in this notebook — the upstream checkpoint supplies the weights, the feature extractor configuration and the label space, and the carried pipeline module adds snapshot verification, input validation, resampling, a fixed output contract and the `validate_inputs` and `evaluation_report` helpers. The default sample is a synthetic tone generated in code; its ranking is demonstration (plumbing) evidence, not a production-quality or benchmark claim.

**Learning objectives:** install the pinned runtime, read what the carried pipeline module guarantees, resolve and digest-verify the immutable upstream model revision, generate a synthetic tone and validate it into an input manifest, run the supported task, read multi-label sigmoid scores correctly (no threshold, not probabilities), exercise an optional BYOD WAV path, produce an evaluation report that is honestly `not-measurable` and says what labelled audio would make the task measurable, and export machine-readable outputs plus provenance.

**This notebook does not demonstrate:** speech transcription, speaker identification, temporal localisation of events inside the window, source separation, audio generation, or any training. The label space is fixed to the 527 AudioSet classes; a sound outside that space still receives ranked labels.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU and uses CUDA automatically when available; inference is float32 on both. The pinned `torch==2.14.0` install and the 346 MB checkpoint are the largest downloads of the run.
- **Knowledge:** basic Python and NumPy; what a mel/filterbank spectrogram is, and why an independent sigmoid per label is not a probability distribution.
- **Data:** the default sample is a deterministic 3 s, 440 Hz sine tone generated in code at 16 kHz, so nothing is downloaded and no private data is needed. Optional BYOD upload is gated off by default so the sample path can run top-to-bottom without interaction. Expected BYOD input: one PCM WAV file (8/16/32-bit), mono or stereo, between 0.025 s and 120 s; it is decoded with the standard-library `wave` module, scaled to `[-1, 1]`, averaged to mono, and resampled to 16 kHz by the pipeline. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded inputs remain in the notebook runtime; this pipeline does not send them to a third-party inference API.
- **External access:** the Hugging Face Hub only, to fetch the pinned `MIT/ast-finetuned-audioset-10-10-0.4593` snapshot (~346 MB) at revision `f826b80d2822…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `torchaudio`, `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchaudio==2.11.0',
    'transformers==4.57.6',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'huggingface-hub==0.36.2',
]
NOTEBOOK_SOURCE = {
    'repository': 'ast-audio-classification-pipeline',
    'repository_revision': '98e26f9d5e483bed938c4e3264577673ae6e4e7e',
    'embedded_module': 'src/ast_audio_classification_pipeline/pipeline.py',
    'module_sha256': '746170ae96560ff754a378d81b8f612c7eddcb4abbb1551f1d1f4ad5c7be45a9',
    'generator': 'build_notebook.py/1',
    'notebook_spec': '1.1',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, torchaudio, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'torchaudio': torchaudio.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/ast_audio_classification_pipeline/pipeline.py` @ `98e26f9d5e48`)

This cell **is** the repository's pipeline module: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the module's, byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (currently 1: the default weights directory becomes working-directory-relative because a notebook has no `__file__`). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever this cell and the module diverge, so what you run here is what the repository tests. Nothing in this cell runs a model yet.

In [ ]:
from __future__ import annotations

import hashlib
import json
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np

MODEL_ID = "MIT/ast-finetuned-audioset-10-10-0.4593"
MODEL_REVISION = "f826b80d28226b62986cc218e5cec390b1096902"
MODEL_LICENSE = "bsd-3-clause"
MODEL_KEY = "ast-audioset"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# The pinned checkpoint expects 16 kHz mono float32. Its feature extractor computes a 128-bin Kaldi
# fbank with a 10 ms hop and pads or crops to 1024 frames, so only the first 10.24 s of audio reach
# the model; anything longer is cropped and reported as `truncated`.
SAMPLE_RATE = 16_000
WINDOW_FRAMES = 1024
MAX_AUDIO_SECONDS = WINDOW_FRAMES * 0.010  # 10.24 s model window
MAX_INPUT_SECONDS = 120.0  # hard ceiling: longer input is rejected, the caller must chunk
MIN_AUDIO_SECONDS = 0.025  # one 25 ms fbank frame
NUM_LABELS = 527  # AudioSet ontology classes in the pinned config.json
DEFAULT_TOP_K = 5
ACTIVATION = "sigmoid"


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check the local snapshot against its DIMER manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = hashlib.sha256()
        with open(file_path, "rb") as fh:
            for chunk in iter(lambda: fh.read(1 << 20), b""):
                digest.update(chunk)
        if digest.hexdigest() != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest.hexdigest()} != manifest {entry['sha256']}")
    return manifest


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def _resample(waveform: np.ndarray, sample_rate: int) -> np.ndarray:
    import torch
    import torchaudio.functional as af

    resampled = af.resample(torch.from_numpy(waveform), orig_freq=sample_rate, new_freq=SAMPLE_RATE)
    return resampled.numpy().astype(np.float32)


INPUT_SCHEMA: dict[str, Any] = {
    "input": "1-D float numpy array of mono samples in [-1, 1], plus the sample_rate it was captured at",
    "sample_rate_hz": f"any positive int; resampled to SAMPLE_RATE={SAMPLE_RATE} when it differs",
    "duration_seconds": [MIN_AUDIO_SECONDS, MAX_INPUT_SECONDS],
    "model_window_seconds": MAX_AUDIO_SECONDS,
    "top_k": [1, NUM_LABELS],
    "preprocessing": (
        f"resample to {SAMPLE_RATE} Hz when needed, 128-bin Kaldi fbank with a 10 ms hop, "
        f"pad or crop to {WINDOW_FRAMES} frames ({MAX_AUDIO_SECONDS} s)"
    ),
}


def _check_inputs(audio: Any, sample_rate: Any, top_k: Any) -> float:
    """Raise TypeError/ValueError naming the first violated ceiling; return the clip duration in seconds."""
    if not isinstance(audio, np.ndarray):
        raise TypeError(f"audio must be a numpy.ndarray, got {type(audio).__name__}")
    if audio.ndim != 1:
        raise ValueError(f"audio must be 1-D mono, got shape {audio.shape}")
    if not np.issubdtype(audio.dtype, np.floating):
        raise TypeError(f"audio must be a float array in [-1, 1], got dtype {audio.dtype}")
    if not isinstance(sample_rate, int) or isinstance(sample_rate, bool) or sample_rate <= 0:
        raise TypeError("sample_rate must be a positive int (the rate the audio was captured at)")
    if not isinstance(top_k, int) or isinstance(top_k, bool) or not 1 <= top_k <= NUM_LABELS:
        raise ValueError(f"top_k must be an int in [1, {NUM_LABELS}]")
    if not np.all(np.isfinite(audio)):
        raise ValueError("audio contains NaN or inf samples")
    duration = audio.shape[0] / sample_rate
    if duration < MIN_AUDIO_SECONDS:
        raise ValueError(f"audio is {duration:.4f} s; minimum is {MIN_AUDIO_SECONDS} s")
    if duration > MAX_INPUT_SECONDS:
        raise ValueError(f"audio is {duration:.2f} s; ceiling is {MAX_INPUT_SECONDS} s (chunk it first)")
    return duration


def validate_inputs(
    waveforms: Any,
    sample_rate: Any,
    *,
    top_k: int = DEFAULT_TOP_K,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, per-clip observations, verdict).

    The checks are the ones ``predict`` applies, through the same private ``_check_inputs``, so a
    rejection here raises exactly what ``predict`` would; a caller that wants the finding recorded
    catches the exception and stores ``str(exc)`` under ``findings``.
    """
    batch = [waveforms] if isinstance(waveforms, np.ndarray) else waveforms
    if not isinstance(batch, Sequence) or isinstance(batch, str | bytes):
        raise TypeError("waveforms must be a 1-D numpy.ndarray or a sequence of them")
    if len(batch) < 1:
        raise ValueError("at least one waveform is required")
    if names is not None and len(names) != len(batch):
        raise ValueError("names must have one entry per waveform")
    inputs = []
    for index, waveform in enumerate(batch):
        duration = _check_inputs(waveform, sample_rate, top_k)
        inputs.append(
            {
                "id": names[index] if names else f"clip-{index}",
                "samples": int(waveform.shape[0]),
                "dtype": str(waveform.dtype),
                "duration_seconds": round(duration, 4),
                "peak_amplitude": round(float(np.max(np.abs(waveform))), 6),
                "will_resample": sample_rate != SAMPLE_RATE,
                "will_truncate": duration > MAX_AUDIO_SECONDS,
            }
        )
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": inputs,
        "sample_rate": sample_rate,
        "top_k": top_k,
        "n_clips": len(inputs),
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    result: Mapping[str, Any], targets: Sequence[Any] | None = None, *, sample_kind: str = "synthetic"
) -> dict[str, Any]:
    """Evaluation stage: always ``not-measurable`` — this repository ships no metric helper (EVAL9).

    Multi-label AudioSet scores need clips labelled against the same 527-label ontology to mean
    anything, and the tutorial sample is a synthetic tone with no ground truth. ``targets`` is
    accepted so the signature matches the fleet's other pipelines, but there is no metric helper to
    route them through: the report stays ``not-measurable`` and names what a real evaluation needs.
    """
    predictions = result["predictions"]
    reason = (
        "no AudioSet-labelled ground truth exists for the evaluated clip, and no metric helper is shipped"
    )
    if targets is not None:
        reason = (
            "targets were supplied, but this repository ships no metric helper for multi-label audio; "
            "score them with an AudioSet-convention mAP implementation of your own"
        )
    return {
        "task": "multi-label audio event classification over the 527 AudioSet labels",
        "score_semantics": (
            f"independent {result.get('activation', ACTIVATION)} score per label: the scores do not sum "
            "to one, several labels can be high at once, none is a calibrated probability, and no "
            "threshold is shipped"
        ),
        "sample_kind": sample_kind,
        "n_clips": 1,
        "n_scored_labels": len(predictions),
        "metrics": [],
        "baselines": [],
        "verdict": "not-measurable",
        "reason": reason,
        "needs": (
            f"clips labelled against the same {NUM_LABELS}-label AudioSet ontology, scored over the full "
            "score vector (predict(..., top_k=527)) with mean average precision plus per-label precision "
            "and recall at a threshold chosen on your own labelled clips; the '0.4593' in the checkpoint "
            "name is the upstream-reported AudioSet mAP and is not measured here"
        ),
        "truncated": bool(result.get("truncated", False)),
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


@dataclass
class ASTAudioClassificationPipeline:
    """Multi-label AudioSet classifier. `_runner` maps a 16 kHz float32 waveform to raw logits (527,)."""

    _runner: Callable[[np.ndarray], np.ndarray]
    labels: list[str]
    device: str

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> ASTAudioClassificationPipeline:
        import torch
        from transformers import ASTFeatureExtractor, ASTForAudioClassification

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            source, kwargs = str(root), dict(local_files_only=True)
        elif allow_download:
            source, kwargs = MODEL_ID, dict(revision=MODEL_REVISION)
        else:
            raise FileNotFoundError(f"no verified snapshot at {root} and allow_download=False")
        extractor = ASTFeatureExtractor.from_pretrained(source, trust_remote_code=False, **kwargs)
        model = ASTForAudioClassification.from_pretrained(
            source, trust_remote_code=False, dtype=torch.float32, **kwargs
        )
        model = model.to(resolved_device).eval()
        labels = [model.config.id2label[i] for i in range(model.config.num_labels)]

        def runner(waveform: np.ndarray) -> np.ndarray:
            inputs = extractor(waveform, sampling_rate=SAMPLE_RATE, return_tensors="pt")
            with torch.inference_mode():
                logits = model(inputs["input_values"].to(resolved_device)).logits
            return logits[0].float().cpu().numpy()

        return cls(runner, labels, resolved_device)

    def predict(
        self,
        audio: np.ndarray,
        sample_rate: int,
        top_k: int = DEFAULT_TOP_K,
    ) -> dict[str, Any]:
        """Classify one clip. `audio` is a 1-D float array; `sample_rate` is the rate it was captured at."""
        duration = _check_inputs(audio, sample_rate, top_k)
        waveform = audio.astype(np.float32, copy=False)
        resampled = sample_rate != SAMPLE_RATE
        if resampled:
            waveform = _resample(waveform, sample_rate)
        logits = np.asarray(self._runner(waveform), dtype=np.float32)
        if logits.shape != (len(self.labels),):
            raise RuntimeError(f"backend returned logits {logits.shape}, expected ({len(self.labels)},)")
        scores = 1.0 / (1.0 + np.exp(-logits))
        order = np.argsort(-scores)[:top_k]
        return {
            "predictions": [
                {"label": self.labels[int(i)], "index": int(i), "score": float(scores[i])} for i in order
            ],
            "activation": ACTIVATION,
            "truncated": duration > MAX_AUDIO_SECONDS,
            "duration_seconds": duration,
            "window_seconds": MAX_AUDIO_SECONDS,
            "resampled": resampled,
            "input_sample_rate": sample_rate,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `4`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `f826b80d2822…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `ASTAudioClassificationPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "ast-audioset",
  "modelId": "MIT/ast-finetuned-audioset-10-10-0.4593",
  "revision": "f826b80d28226b62986cc218e5cec390b1096902",
  "files": [
    {
      "path": "README.md",
      "bytes": 1165,
      "sha256": "dbc8ce1fc5abd1635d073640d8cb032901bfe264a5e06507f363f10a6112cfc8"
    },
    {
      "path": "config.json",
      "bytes": 26763,
      "sha256": "a93d525511d77e8ecc933d09674b85099815bbbb417c228a4edd655e252fb9ff"
    },
    {
      "path": "model.safetensors",
      "bytes": 346404948,
      "sha256": "ae0c1e2ad4e1381d851fa9bf298ba13ebc9c5a914cdee2dbe427a6583869924d"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 297,
      "sha256": "8d04ba5a9c6fca5d39d0de2b1fd05ecf79deb589fbba279728bbebac39934231"
    }
  ],
  "totalBytes": 346433173
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
print({'verified_files': [entry['path'] for entry in snapshot.get('files', [])]})
pipe = ASTAudioClassificationPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Generate the synthetic sample or optional BYOD

The default sample is **synthetic**: a deterministic 3 s, 440 Hz sine tone at amplitude 0.5, generated in code at the model's 16 kHz rate as a float32 mono array (the same kind of input the repository's smoke run used), so it needs no download and its SHA-256 is printed for the record. A pure tone is not a recording of any real acoustic event, so it has **no ground truth** and whatever ranking the model returns is a sanity check that the input contract, feature extraction and forward pass work — not a correctness measurement. BYOD is optional and disabled by default; when enabled, upload one PCM WAV file. It is decoded with the standard-library `wave` module (no extra decoder is pinned), integer samples are scaled to `[-1, 1]`, stereo is averaged to mono, and the original sample rate is passed to the pipeline, which resamples to 16 kHz with `torchaudio.functional.resample` — resampling cannot restore content above the original Nyquist frequency.

In [ ]:
import hashlib
import io
import wave

import numpy as np

USE_BYOD = False  # @param {type:"boolean"}
TONE_SECONDS = 3.0
TONE_HZ = 440.0
TONE_AMPLITUDE = 0.5

if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    clip_name = next(iter(uploaded))
    with wave.open(io.BytesIO(uploaded[clip_name]), 'rb') as handle:
        channels, sample_width, sample_rate, frames = handle.getnchannels(), handle.getsampwidth(), handle.getframerate(), handle.getnframes()
        raw = handle.readframes(frames)
    if sample_width == 1:
        samples = (np.frombuffer(raw, dtype=np.uint8).astype(np.float32) - 128.0) / 128.0
    elif sample_width == 2:
        samples = np.frombuffer(raw, dtype='<i2').astype(np.float32) / 32768.0
    elif sample_width == 4:
        samples = np.frombuffer(raw, dtype='<i4').astype(np.float32) / 2147483648.0
    else:
        raise ValueError(f'{clip_name}: {8 * sample_width}-bit PCM is not supported here; convert the file to 16-bit PCM WAV and rerun this cell.')
    audio = samples.reshape(-1, channels).mean(axis=1).astype(np.float32) if channels > 1 else samples
    sample_kind = 'BYOD'
else:
    # Deterministic synthetic tone: no randomness, so no seed is needed and the digest is stable.
    sample_rate = SAMPLE_RATE
    t = np.arange(int(TONE_SECONDS * sample_rate)) / sample_rate
    audio = (TONE_AMPLITUDE * np.sin(2 * np.pi * TONE_HZ * t)).astype(np.float32)
    clip_name = f'synthetic_sine_{int(TONE_HZ)}hz_{int(TONE_SECONDS)}s.wav'
    sample_kind = 'synthetic'

audio_sha256 = hashlib.sha256(audio.tobytes()).hexdigest()
print({'sample_kind': sample_kind, 'name': clip_name, 'samples': int(audio.shape[0]), 'sample_rate': sample_rate, 'dtype': str(audio.dtype), 'float32_sha256': audio_sha256})

## 5. Validate the clip → input manifest

`validate_inputs` is the pipeline's public validation stage: it applies exactly the checks `predict` applies, through the same private check, so the two cannot diverge — 1-D float array, positive integer sample rate, finite samples, duration between `MIN_AUDIO_SECONDS` and `MAX_INPUT_SECONDS`, `top_k` in 1..`NUM_LABELS`. It returns an **input manifest** naming the schema and ceilings, each clip's identifier, sample count, duration, peak amplitude, and whether it will be resampled or truncated, written to `outputs/ast_audio_classification_input_manifest.json`. The ceilings are printed first, before any model work. A clip above `MAX_INPUT_SECONDS` (120 s) is **rejected** — chunk it first — while a clip longer than the 10.24 s model window is **accepted** but **only its first 10.24 s reach the model**: the manifest flags `will_truncate` and the result carries `truncated: True`. Clips shorter than one 25 ms filterbank frame are rejected. To show what rejection looks like, the cell also validates a deliberately over-long clip and records the pipeline's own error message as a finding. Nothing else is dropped or altered.

In [ ]:
import json
import os

os.makedirs('outputs', exist_ok=True)
print({'ceilings': {'SAMPLE_RATE': SAMPLE_RATE, 'MIN_AUDIO_SECONDS': MIN_AUDIO_SECONDS, 'MAX_AUDIO_SECONDS': MAX_AUDIO_SECONDS, 'MAX_INPUT_SECONDS': MAX_INPUT_SECONDS, 'NUM_LABELS': NUM_LABELS}})
input_manifest = validate_inputs(audio, sample_rate, top_k=5, names=[clip_name])
# Demonstrate rejection on a clip that breaks a ceiling; the finding is recorded, not swallowed.
try:
    validate_inputs(np.zeros(int((MAX_INPUT_SECONDS + 1) * SAMPLE_RATE), dtype=np.float32), SAMPLE_RATE)
except ValueError as exc:
    input_manifest['findings'].append({'input': 'over-long-probe', 'verdict': 'rejected', 'message': str(exc)})
if input_manifest['inputs'][0]['will_truncate']:
    print(f"NOTE: {clip_name} is longer than the {MAX_AUDIO_SECONDS} s model window; only its first {MAX_AUDIO_SECONDS} s reach the model and the result is flagged truncated.")
with open('outputs/ast_audio_classification_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print(json.dumps(input_manifest, indent=2))

## 6. Classify and read the scores correctly

`predict` returns a `predictions` list of `{label, index, score}` entries **ordered by descending score** — rank position is the label ordering, and the exported files preserve it — plus `activation` (`sigmoid`), `truncated`, `duration_seconds`, `window_seconds`, `resampled`, `input_sample_rate`, and the model identity. Each `score` is an **independent sigmoid** of that label's logit: this is multi-label classification, so the scores do not sum to one, several labels can be high at once, and a score is **not a calibrated probability** (the head was trained with binary cross-entropy on weak, incomplete AudioSet labels). **No decision threshold is shipped**: `top_k` (default 5, ceiling 527) is a presentation choice, not an acceptance rule, and no label is asserted present or absent. The operator owns the threshold and should set it per label from precision-recall curves on their own labelled clips. Inference is deterministic on a fixed device and dtype (no sampling, `model.eval()`, `torch.inference_mode`); CUDA kernel selection and the resampler can shift scores in the third or fourth decimal place across hardware. As recorded in the model card, the repository's smoke run on this same tone (CUDA, float32, verified snapshot) ranked `Sine wave` first at score 0.84; that is one observation for a synthetic tone and sanity evidence only — a materially different top label on your runtime is a signal to check the install, not a measurement of anything.

In [ ]:
result = pipe.predict(audio, sample_rate=sample_rate, top_k=5)
print({'activation': result['activation'], 'truncated': result['truncated'], 'resampled': result['resampled'], 'duration_seconds': round(result['duration_seconds'], 3), 'window_seconds': result['window_seconds'], 'input_sample_rate': result['input_sample_rate'], 'device': pipe.device})
for rank, item in enumerate(result['predictions'], start=1):
    print(f"{rank:>2}. index {item['index']:>3}  score {item['score']:.4f}  {item['label']}")

## 7. Evaluate → evaluation report

`evaluation_report` is the pipeline's public evaluation stage and always produces a report. Its verdict here is **always `not-measurable`**: this repository ships no metric helper, and a synthetic tone has no ground truth, so there is nothing honest to report as a number. The report names the score semantics (independent sigmoids, not probabilities, no threshold) and states what a real evaluation needs: clips labelled against the same 527-label ontology, the full score vector (`predict(..., top_k=527)`), and mean average precision plus per-label precision and recall at a threshold chosen on your own labelled clips. The `0.4593` in the checkpoint name is the upstream-reported AudioSet mAP for this configuration and is not measured here. The report is written to `outputs/ast_audio_classification_evaluation_report.json`.

In [ ]:
report = evaluation_report(result, sample_kind=sample_kind)
with open('outputs/ast_audio_classification_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
print(json.dumps(report, indent=2))
if report['verdict'] == 'not-measurable':
    print('No metric is computed: the pipeline ships no metric helper and the sample has no ground truth; the ranking above is sanity evidence only.')

## 8. Export outputs and provenance

Machine-readable JSON preserves the full result (rank-ordered sigmoid scores, activation, truncation and resampling flags), the input manifest, the evaluation report, the clip identity and digest, the notebook's source (repository, revision, embedded module digest, generator), the model identifier, the immutable model revision, the model licence, and the runtime identity (Python, `torch`, `torchaudio`, `transformers`, device). The rank-ordered top-k table is also written as CSV with explicit `clip`, `rank`, `index`, `label` and `score` columns so label ordering survives downstream use. No credentials are recorded.

In [ ]:
import csv

payload = {
    'prediction': result,
    'evaluation_report': report,
    'input_manifest': input_manifest,
    'sample': {'kind': sample_kind, 'name': clip_name, 'samples': int(audio.shape[0]), 'sample_rate': sample_rate, 'float32_sha256': audio_sha256},
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'torchaudio': torchaudio.__version__,
        'transformers': transformers.__version__,
        'device': pipe.device,
    },
}
with open('outputs/ast_audio_classification_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
with open('outputs/ast_audio_classification_top_k.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.writer(handle)
    writer.writerow(['clip', 'rank', 'index', 'label', 'score'])
    for rank, item in enumerate(result['predictions'], start=1):
        writer.writerow([clip_name, rank, item['index'], item['label'], f"{item['score']:.6f}"])
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The ranked labels are independent sigmoid scores over the fixed 527-label AudioSet ontology; they are not probabilities of presence, they do not sum to one, and the pipeline ships no threshold, so nothing in this notebook asserts that an event is present or absent. On the synthetic tone the ranking is sanity evidence by construction and the evaluation report says `not-measurable`; a ranking shown for a BYOD clip is a single-clip observation for that recording and must not be generalized to a domain, microphone, or acoustic environment. Only the first 10.24 s of a clip reach the model, resampling from other rates loses content above the original Nyquist frequency, and sounds outside the ontology still receive some ranked label. The pipeline provides no transcription, speaker identity, temporal localisation, source separation, or training capability.

Successful execution proves that the recorded repository revision's pipeline module, carried in this notebook, can acquire and digest-verify the pinned model, validate the demonstrated input, execute the public pipeline path, and emit the shown machine-readable outputs in the tested runtime — without the repository being reachable. It does **not** establish benchmark superiority, deployment calibration, safety for high-consequence decisions, or production fitness on an unseen domain.

**Next experiments:** enable `USE_BYOD` with a short real recording (a door closing, a dog barking) and compare how the sigmoid scores spread across related ontology labels; change `TONE_HZ` and watch which tone-like labels (`Sine wave`, `Dial tone`, `Beep, bleep`) move; upload a clip longer than 10.24 s and confirm that the input manifest flags `will_truncate` and the result carries `truncated: True`, then chunk it yourself and score each window separately.

## References

- Repository README: https://github.com/kurtvalcorza/ast-audio-classification-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/ast-audio-classification-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/ast-audio-classification-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/MIT/ast-finetuned-audioset-10-10-0.4593
- Upstream code: https://github.com/YuanGongND/ast
- AST: Audio Spectrogram Transformer (Gong, Chung, Glass, 2021): https://arxiv.org/abs/2104.01778
- AudioSet ontology: https://research.google.com/audioset/